### Gathering Historical Fantasy Scores

In the cells below I will be gathering historical fantasy football players points for a specified season. I will first look at the 2025 season and perform transformations, cleaning, and EDA to determine if more data is needed and identify holes. This data is useful because we can use the historical points as a way to compare out models output performance as in the accuracy of our output.

This data is gathered from ESPN's default-leage endpoint api: [Fantasy API 2025](https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/seasons/2025)

In [61]:
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import os
#root directory of project
root = Path.cwd().parent
data_path = root/"data"
import nflreadpy as nfl
import polars as pl
import sys


sys.path.append(str(Path("../src").resolve()))   # folder that CONTAINS the .py file

import utils
from utils import write_to_dict, update_data 

In [62]:
# def write_to_dict(df, name):
#     '''Update dictionary csv files '''
#     columns = df.columns.to_numpy()
#     dict_ = pd.DataFrame(columns)
#     try:
#         dict_.to_csv(data_path/f"dicts/{name}_dictionary.csv")
#     except Exception as e:
#         print(f"failed to write dictionary with exception {e}")

# def update_data(df, name):
#     try:
#         df.to_csv(data_path/f"{name}_final.csv")
#     except Exception as e:
#         print(f"failed to updata data: {e}")

In [99]:
Season = 2026

In [100]:
## load fantasy historical data
weekly_df = pd.read_csv(data_path/f"player_weeks_{Season}.csv")
#weekly_df.head()


In [101]:
# load player summary data
player_sum = pd.read_csv(data_path/f"player_summary_{Season}.csv")
#player_sum.head()

In [102]:
write_to_dict(player_sum, f"player_sum_{Season}")


In [103]:
# load player log data
player_logs = pd.read_csv(data_path/f"player_gamelogs_{Season}.csv")
#player_logs.head()


In [104]:
write_to_dict(player_logs, f"player_logs_{Season}")

In [105]:
display(weekly_df.shape)
display(player_sum.shape)
display(player_logs.shape)

(12861, 7)

(636, 84)

(4440, 59)

In [106]:
display(weekly_df["player_id"].nunique())
display(player_sum["player_id"].nunique())
display(player_logs["player_id"].nunique())

792

636

636

In [107]:
## match our historical fantasy players with our other data
players_in_sum = player_sum["player_id"].unique()
players_set = set(players_in_sum)

players_in_logs = player_logs["player_id"].unique()
players_set_2 = set(players_in_logs)

assert players_set == players_set_2, "not same players"

In [108]:
# now that two of our df contain the same players, we only need to filter the historical fantasy data to contain the player_id as the other dataframes

weekly_df = weekly_df[weekly_df["player_id"].isin(players_set)]
weekly_df.shape

(10664, 7)

## cleaning missing values in projected points

There are some cases where a player was not set to play but ended up earning fantasy points and hence missing values for the projected points. So if a player has missing projected points and scored fantasy points, I am going to impute the projected points with either the median or mean of their fantasy points of that season, I will use the player summary dataframe to calculate these statistics and then determine which of the central tendencies I would like to use

In [109]:
mask = weekly_df[weekly_df["projected_points"].isna() & weekly_df["fantasy_points"].isna()]

# check why there are no columns with both no point and no projected points
no_proj = weekly_df[weekly_df["projected_points"].isna()]
no_pts = weekly_df[weekly_df["fantasy_points"].isna()]

display(mask.head())
display(no_proj.head())
display(no_pts.head())

print("length of no projected points:", len(no_proj))
print("length of no proj with week = 1:", len(no_proj[no_proj["week"] == 1]))
print(len(no_pts[no_pts["projected_points"] == 0]))

,player_id,player,position,pro_team_id,week,fantasy_points,projected_points


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
153,4569682,Audric Estime,RB,18,1,0.0,NaN
273,4426386,Roschon Johnson,RB,3,1,0.0,NaN
298,4431466,Jordan Watkins,WR,25,1,0.0,NaN
373,4698253,Lan Larison,RB,17,1,2.9,NaN
407,4429582,Joe Fagnano,QB,33,1,0.0,NaN


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
301,4360698,Joe Milton,QB,21,1,NaN,0.0
343,5208518,Charlie Smyth,K,18,1,NaN,0.0
349,16339,Brandon McManus,K,14,1,NaN,0.0
351,4685279,Jaydon Blue,RB,0,1,NaN,0.0
402,3930066,Van Jefferson,WR,28,1,NaN,0.0


length of no projected points: 430
length of no proj with week = 1: 57
2479


In [110]:
# in player summary we already have average fantasy points so I will compute the mean 

weekly_df["med_proj_pts"]  = weekly_df.groupby("player_id")["projected_points"].transform("median")
#weekly_df.head()

In [111]:
weekly_df["projected_points"] = np.where(
    weekly_df["projected_points"].isna(),
    weekly_df["med_proj_pts"],
    weekly_df["projected_points"]
)

#weekly_df.head()

# Data Integration
From looking at the Data dictionaries, our summary and player logs contain the fantasy data already so I will Merge the logs with the ```weekly_df``` on player_id and week

In [112]:
#key columns are columns we merge on
key_cols = {"player_id","week"}

#find set of columns that overlap excluding key columns
overlapping_cols = (set(player_logs.columns) & set(weekly_df.columns)) - key_cols


#overlapping_cols

#left merge on key columns, drop overlapping columns
merged_df = pd.merge(player_logs, weekly_df.drop(columns=list(overlapping_cols)), on=list(key_cols), how="left")

#merged_df.head()

In [113]:
merged_df["team"].nunique()
merged_df["season_type"].nunique()

if merged_df["season_type"].nunique() == 1:
    merged_df = merged_df.drop(columns = ["pro_team_id", "season_type"])
#we have team name and team ID number
# merged_df.drop(columns = "pro_team_id")

#merged_df.head()

## Adding in Weather Data
Using ```nflready``` library, we can gather the weather location based on the value in the ```eventId``` column and use that to append corresponding weather data to the rows. 

In [78]:



sched = nfl.load_schedules(seasons=Season)  
weather = sched.select([
    "espn", "game_id", "gameday", "gametime",
    "stadium", "roof", "surface", "temp", "wind"
])
weather = weather.to_pandas()
# your_df: your ESPN data with an event ID column
merged_df["eventId"] = merged_df["eventId"].astype(str)
df = merged_df.merge(weather, left_on="eventId", right_on="espn", how="left")
#df.head()

### Updating temperature data
We are going to impute the temperature missing values for domes to be the average temperature of a football dome stadium which happens to be around 70 farhenhiet.

In [115]:
# fixing roof data
df["roof"].unique()

df["roof"] = np.where(df["roof"] == "closed",
                      "dome", 
                      df["roof"])

df["roof"].head()

0    outdoors
1        dome
2        dome
3        dome
4    outdoors
Name: roof, dtype: str

In [116]:
df["roof"].unique()

<ArrowStringArray>
['outdoors', 'dome', 'open', nan]
Length: 4, dtype: str

In [117]:
# impute mean indoor stadium temperature when theres a roof
df["temp"] = np.where(
    df["roof"] == "dome",
    70,
    df["temp"])

#there shouldnt be wind in a dome
df["wind"] = np.where(
    (df["wind"].isna()) & (df["roof"] == "dome"),
    0,
    df["wind"]
)


# finally, impute the outdoor wind values with the mean wind value outdoors
mean_wind = df[df["roof"] == "outdoors"]["wind"].mean()

df["wind"] = np.where(
    (df["wind"].isna()) & (df["roof"] == "outdoors"),
    mean_wind,
    df["wind"])



In [118]:
df["stadium"].unique()

<ArrowStringArray>
[              'Hard Rock Stadium',                      'Ford Field',
              'State Farm Stadium',         'Bank of America Stadium',
                'Highmark Stadium',                        'Bernabeu',
                  'Paycor Stadium',                'M&T Bank Stadium',
                'Acrisure Stadium',                 'Reliant Stadium',
               'U.S. Bank Stadium',               'Allegiant Stadium',
                    'SoFi Stadium',                     'Lumen Field',
      'Empower Field at Mile High',               'Northwest Stadium',
               'Lucas Oil Stadium',                    'AT&T Stadium',
         'Lincoln Financial Field',                   'Lambeau Field',
                'Maracana Stadium',                 'MetLife Stadium',
                  'Levi's Stadium',           'Mercedes-Benz Stadium',
        'Melbourne Cricket Ground',           'Raymond James Stadium',
               'Caesars Superdome', 'GEHA Field at Arrowhe

In [119]:
#fixing missing surface
df.loc[df["stadium"] == "SoFi Stadium","surface"] = "matrixturf"



## Adding in Location Data
I would also like to add Location data to this data set using the stadium name

In [120]:
h_a = {"@": "away", "vs": "home"}

df["atVs"] = df["atVs"].map(h_a)

#df.head()

In [121]:

stadium_locations = {
    'Soldier Field': 'Chicago,IL',
    'U.S. Bank Stadium': 'Minneapolis,MN',
    'Ford Field': 'Detroit,MI',
    'SoFi Stadium': 'Inglewood,CA',
    'Lincoln Financial Field': 'Philadelphia,PA',
    'FedExField': 'Landover,MD',
    'GEHA Field at Arrowhead Stadium': 'Kansas City,MO',
    'Paycor Stadium': 'Cincinnati,OH',
    'M&T Bank Stadium': 'Baltimore,MD',
    'Lambeau Field': 'Green Bay,WI',
    "Levi's Stadium": 'Santa Clara,CA',
    'Lucas Oil Stadium': 'Indianapolis,IN',
    'FirstEnergy Stadium': 'Cleveland,OH',
    'State Farm Stadium': 'Glendale,AZ',
    'MetLife Stadium': 'East Rutherford,NJ',
    'NRG Stadium': 'Houston,TX',
    'Raymond James Stadium': 'Tampa,FL',
    'Mercedes-Benz Superdome': 'New Orleans,LA',
    'Lumen Field': 'Seattle,WA',
    'Hard Rock Stadium': 'Miami Gardens,FL',
    'New Era Field': 'Orchard Park,NY',
    'Acrisure Stadium': 'Pittsburgh,PA',
    'Empower Field at Mile High': 'Denver,CO',
    'Mercedes-Benz Stadium': 'Atlanta,GA',
    'Bank of America Stadium': 'Charlotte,NC',
    'Nissan Stadium': 'Nashville,TN',
    'Gillette Stadium': 'Foxborough,MA',
    'AT&T Stadium': 'Arlington,TX',
    'Allegiant Stadium': 'Paradise,NV',
    'TIAA Bank Stadium': 'Jacksonville,FL'
}


df["location"] = df["stadium"].map(stadium_locations)

#redundant colunn

#df.head()

## Adding features
We also need to consider how much playtime a player gets to and whether or not a player actually played as in their health status each game. We should also consider whether they are starters or second string players because that will impact how often they play and ultimately the number of fantasy points they are able to score.

In [122]:
df.columns

Index(['player_id', 'player', 'position', 'team', 'season', 'season_type',
       'week', 'eventId', 'date', 'opponent', 'atVs', 'result', 'score',
       'rushingAttempts', 'rushingYards', 'yardsPerRushAttempt',
       'rushingTouchdowns', 'longRushing', 'receptions', 'receivingTargets',
       'receivingYards', 'yardsPerReception', 'receivingTouchdowns',
       'longReception', 'fumbles', 'fumblesLost', 'fumblesForced',
       'kicksBlocked', 'completions', 'passingAttempts', 'passingYards',
       'completionPct', 'yardsPerPassAttempt', 'passingTouchdowns',
       'interceptions', 'longPassing', 'sacks', 'QBRating', 'adjQBR',
       'longFieldGoalMade', 'fieldGoalPct', 'fieldGoalsMadeYardsAverage',
       'totalKickingPoints', 'fieldGoalsMade1_19', 'fieldGoalAttempts1_19',
       'fieldGoalsMade20_29', 'fieldGoalAttempts20_29', 'fieldGoalsMade30_39',
       'fieldGoalAttempts30_39', 'fieldGoalsMade40_49',
       'fieldGoalAttempts40_49', 'fieldGoalsMade50', 'fieldGoalAttempts50',
  

In [ ]:

df = df.copy()
df["date"] = pd.to_datetime(df["date"])

# Aggregate player statistics into team-game statistics.
team_games = (
    df.groupby(["season", "team", "game_id", "date"], as_index=False)
      .agg(
          team_pass_attempts=("passingAttempts", "sum"),
          team_passing_yards=("passingYards", "sum"),
          team_passing_tds=("passingTouchdowns", "sum"),
          team_rushing_attempts=("rushingAttempts", "sum"),
          team_rushing_yards=("rushingYards", "sum"),
      )
      .sort_values(["team", "date"])
)

# Calculate historical averages using prior games only.
features = [
    "team_pass_attempts",
    "team_passing_yards",
    "team_passing_tds",
    "team_rushing_attempts",
    "team_rushing_yards",
]

for col in features:
    team_games[f"{col}_last5"] = (
        team_games.groupby(["season", "team"])[col]
        .transform(
            lambda s: s.shift(1).rolling(5, min_periods=1).mean()
        )
    )

ValueError: time data "2026-11-15T21:25Z" doesn't match format "%Y-%m-%dT%H:%M:%S.%f%z". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
team_games.head()

,season,team,game_id,date,team_pass_attempts,team_passing_yards,team_passing_tds,team_rushing_attempts,team_rushing_yards,team_pass_attempts_last5,team_passing_yards_last5,team_passing_tds_last5,team_rushing_attempts_last5,team_rushing_yards_last5
0,2025,ARI,2025_01_ARI_NO,2025-09-07 17:00:00+00:00,29.0,163.0,2.0,27.0,146.0,NaN,NaN,NaN,NaN,NaN
1,2025,ARI,2025_02_CAR_ARI,2025-09-14 20:05:00+00:00,25.0,220.0,1.0,22.0,82.0,29.000000,163.000000,2.000000,27.000000,146.000000
2,2025,ARI,2025_03_ARI_SF,2025-09-21 20:25:00+00:00,35.0,159.0,1.0,28.0,106.0,27.000000,191.500000,1.500000,24.500000,114.000000
3,2025,ARI,2025_04_SEA_ARI,2025-09-26 00:15:00+00:00,41.0,200.0,2.0,17.0,89.0,29.666667,180.666667,1.333333,25.666667,111.333333
4,2025,ARI,2025_05_TEN_ARI,2025-10-05 20:05:00+00:00,32.0,220.0,0.0,29.0,168.0,32.500000,185.500000,1.500000,23.500000,105.750000


In [ ]:
# 1. team totals per week (summed over that team's players)
team_week = (
    df.groupby(["season", "team", "week"], as_index=False)
      .agg(pass_yds=("passingYards", "sum"), rush_yds=("rushingYards", "sum"))
)
team_week["played"] = 1

# 2. full season x team x week grid, so bye weeks show up as 0 yards / 0 games
keys = ["season", "team"]
grid_idx = pd.MultiIndex.from_product(
    [team_week["season"].unique(), team_week["team"].unique(), range(1, int(df["week"].max()) + 1)],
    names=["season", "team", "week"],
)
grid = team_week.set_index(["season", "team", "week"]).reindex(grid_idx).fillna(0)

# 3. cumulative totals, shifted one week so each row only sees PRIOR games (no leakage)
cum = grid.groupby(level=keys).cumsum()
prior = cum.groupby(level=keys).shift(1)

# 4. yards per game played (fair across bye weeks), then rank all teams within each week
games = prior["played"].where(prior["played"] > 0)
prior["pass_ypg"] = prior["pass_yds"] / games
prior["rush_ypg"] = prior["rush_yds"] / games

by_week = prior.groupby(level=["season", "week"])
prior["team_pass_rank"] = by_week["pass_ypg"].rank(ascending=False, method="min")
prior["team_rush_rank"] = by_week["rush_ypg"].rank(ascending=False, method="min")

# 5. attach to the player rows (assigning overwrites the columns if they already exist)
ranks = prior[["team_pass_rank", "team_rush_rank"]]
row_keys = pd.MultiIndex.from_frame(df[["season", "team", "week"]])

df["team_pass_rank"] = ranks["team_pass_rank"].reindex(row_keys).to_numpy()
df["team_rush_rank"] = ranks["team_rush_rank"].reindex(row_keys).to_numpy()

In [ ]:
## update our data file
write_to_dict(df, f"df_{Season}", data_path)
update_data(df, f"df_{Season}", data_path)

In [ ]:
df[df['week'] == 10].head()

,player_id,player,position,team,season,week,eventId,date,opponent,atVs,...,gameday,gametime,stadium,roof,surface,temp,wind,location,team_pass_rank,team_rush_rank
8,4429795,Jahmyr Gibbs,RB,DET,2025,10,401772878,2025-11-09 21:25:00+00:00,WSH,away,...,2025-11-09,16:25,FedExField,outdoors,grass,64.0,9.0,"Landover,MD",13.0,9.0
24,3117251,Christian McCaffrey,RB,SF,2025,10,401772879,2025-11-09 21:25:00+00:00,LAR,home,...,2025-11-09,16:25,Levi's Stadium,outdoors,grass,77.0,3.0,"Santa Clara,CA",2.0,26.0
58,4426515,Puka Nacua,WR,LAR,2025,10,401772879,2025-11-09 21:25:00+00:00,SF,away,...,2025-11-09,16:25,Levi's Stadium,outdoors,grass,77.0,3.0,"Santa Clara,CA",4.0,19.0
74,4430807,Bijan Robinson,RB,ATL,2025,10,401772636,2025-11-09 14:30:00+00:00,IND,home,...,2025-11-09,09:30,Lucas Oil Stadium,dome,grass,70.0,2.0,"Indianapolis,IN",17.0,15.0
91,4361307,Trey McBride,TE,ARI,2025,10,401772773,2025-11-09 21:05:00+00:00,SEA,away,...,2025-11-09,16:05,Lumen Field,outdoors,fieldturf,59.0,5.0,"Seattle,WA",18.0,20.0
